# IMU Chronos Grid-Search Results Analysis

Same shape as `results_analysis.ipynb`, but consumes the Chronos-feature variant of the grid-search pickle (`grid_search_results_general_db_imu_chronos_chronos.pkl`). Reports the balanced / LOSO / signant F1 scores side-by-side.

## Configuration

In [ ]:
CHRONOS_RESULT = "grid_search_results_general_db_imu_chronos_chronos.pkl"
FEATURE_NAMES_GLOB = "feature_names*.txt"


## Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
from tabulate import tabulate
import glob
import matplotlib.pyplot as plt
import matplotlib
print(matplotlib.style.available)

## Load feature-name dictionary

In [ ]:
features_names_dict = {}

for features_file in glob.glob(FEATURE_NAMES_GLOB):
    with open(features_file, "r") as f:
        feature_names = f.readlines()
        features_names_dict[features_file.split(".")[0]] = list(filter(lambda x: (not x in ["subject", "label","Unnamed: 0"]),[name.strip() for name in feature_names]))

## Tabulate balanced / LOSO / signant results

In [ ]:

database = CHRONOS_RESULT

name = " ".join(database.split('.')[0].split("_")[3:])
print(f"Results for {name}")
print("="*50)

handle = pickle.load(open(database, 'rb'))

balanced_results = handle["balanced_part"]
loso_results = handle["loso_part"]
signant_results = handle["signant_part"]

balanced_agg = {}
for k,v in balanced_results.items():
    f1_score = v["f1_score"]
    model_name,database = k.split("-")
    database = database.split(".")[0]
    balanced_agg[model_name] = f1_score

print("RANDOM PARTITION\n")

print(tabulate(balanced_agg.items(),["Model","F1 Score"]))
print("--"*30)

loso_results_agg = {}

for k,v in loso_results.items():
    f1_score = v["f1_score"]
    model_name,subject = k.split("-")
    loso_results_agg[model_name] = loso_results_agg.get(model_name, [])
    loso_results_agg[model_name].append(f1_score)

for model_name, scores in loso_results_agg.items():
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    loso_results_agg[model_name] = f"{mean_score:.2f} +- {std_score:.2f}"

print("LOSO RESULTS")
print(tabulate(loso_results_agg.items(),["Model","F1 Score"]))
print("--"*30)
signant_agg = {}
for k,v in signant_results.items():
    f1_score = v["f1_score"]
    model_name,db = k.split("-")
    signant_agg[model_name] = f1_score

print("SIGNANT-NONSIGNANT")
print(tabulate(signant_agg.items(),["Model","F1 Score"]))
print("--"*30)